# 03 · Least Squares and the Failure of Naive Inversion

*Derive weighted least squares and see why excellent fit can mean terrible slip.*

**Learning objectives**

- derive the ordinary and weighted normal equations
- interpret singular values and condition number
- distinguish $\chi^2$ from reduced $\chi^2_\nu$
- diagnose noise amplification in an unregularized solution

**Prerequisites:** Chapters 01–02; basic probability and linear algebra  
**Estimated time:** 60–90 minutes

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

The naive inverse problem has an enticing promise: choose the slip that fits
the data best. Distributed faults expose the flaw in that promise. Deep and
neighboring patches can produce almost the same surface field, so noise can be
fit by large, alternating slip with almost no penalty in data space.


## Theory deepening: WLS and the singular-value picture

Ordinary least squares minimizes $\Phi=(Gm-d)^T(Gm-d)$. Setting its gradient
$2G^T(Gm-d)$ to zero gives the normal equations. With
$\epsilon\sim\mathcal N(0,C_d)$, whitening changes the objective to

$$\Phi=(Gm-d)^TW(Gm-d),\qquad W=C_d^{-1},$$

and the estimator becomes

$$\hat m=(G^TWG)^{-1}G^TWd.$$

Under a correct linear model with zero-mean errors this is the best linear
unbiased estimator (Gauss–Markov). “Best” is conditional on the covariance.

For whitened $G=U\Sigma V^T$, each data coefficient is divided by a singular
value. Small $\sigma_i$ therefore magnifies noise along $v_i$; rank deficiency
sets some $\sigma_i$ to zero. Reduced $\chi^2$ near one checks residual scale
against declared uncertainty, not model uniqueness.


## The linear inverse problem

Data are the forward model plus noise:

$$ \mathbf{d} = G\,\mathbf{m} + \boldsymbol{\varepsilon}, \qquad
\operatorname{cov}(\boldsymbol{\varepsilon}) = C_d. $$

**Ordinary least squares (OLS)** picks the slip that minimizes the squared
misfit $\lVert G\mathbf{m} - \mathbf{d}\rVert^2$. Setting the gradient to zero
gives the *normal equations*:

$$ G^{\mathsf T} G\, \hat{\mathbf{m}} = G^{\mathsf T}\mathbf{d}. $$

**Weighted least squares (WLS)** accounts for the fact that some data are more
precise than others. With the weight matrix $W = C_d^{-1}$,

$$ \hat{\mathbf{m}} = \left(G^{\mathsf T} W G\right)^{-1} G^{\mathsf T} W \mathbf{d}. $$

Down-weighting noisy observations keeps them from dominating the fit. `geodef`
builds $W$ from each dataset's uncertainties automatically, so `invert()`
defaults to WLS.

The catch is the matrix $G^{\mathsf T} W G$. If it is **ill-conditioned** —
nearly singular — then small changes in the data (i.e. noise) cause large
changes in $\hat{\mathbf{m}}$. Deep and adjacent patches produce almost the
same surface signal, so their columns of $G$ are nearly parallel and the
inverse amplifies noise. We will see exactly that below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.data.gnss(lon=glon, lat=glat, east=_zero, north=_zero, up=_zero, sigma_east=_one, sigma_north=_one, sigma_up=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.data.gnss(
    lon=glon, lat=glat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_sta, sigma), sigma_north=np.full(n_sta, sigma), sigma_up=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

## The data we are trying to explain

Here is the true slip we are pretending not to know, and the noisy GNSS
displacements it produced. Our job: recover the left panel from the arrows on
the right.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
geodef.plot.slip(fault, slip_true[N:], ax=ax1, title="True dip-slip (unknown)",
                 coords="geographic",
                 colorbar_label="Slip (m)")
geodef.plot.vectors(gnss, fault, ax=ax2, title="Observed GNSS (with noise)")
plt.tight_layout()

## Solve it: weighted least squares

Because the true slip is pure dip-slip, we solve for one slip component per
patch with `components='dip'` (notebook 08 revisits component choices). With no
`regularization` argument, `invert()` performs a plain weighted least-squares solve
— no regularization, no prior.

In [ ]:
raw = geodef.solve(fault, gnss, components="dip")
print(f"method:  weighted least squares (unregularized)")
print(f"reduced chi^2: {raw.reduced_chi2:.3f}")
print(f"RMS residual:  {raw.rms * 1000:.2f} mm")

A reduced $\chi^2$ near (or below) 1 means the model fits the data to within
their uncertainties. That sounds like success — until we look at the slip.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
lim_true = slip_true[N:].max()
lim_raw = abs(raw.dip_slip).max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, cmap="RdBu_r",
                 vmin=-lim_true, vmax=lim_true, title="True slip",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, raw.dip_slip, ax=ax2, cmap="RdBu_r",
                 vmin=-lim_raw, vmax=lim_raw, title="Unregularized WLS",
                 colorbar_label="Slip (m)")
plt.tight_layout()

## Overfitting: a "perfect" fit to nonsense

The recovered slip is wild — swinging positive and negative by far more than the
true peak, with a checkerboard pattern that has no physical meaning. Yet it fits
the data slightly *better* than the truth would. It is fitting the **noise**.

Compare the numbers:

In [ ]:
def roughness(m):
    return float(np.std(np.diff(m)))

print(f"true slip:    peak {slip_true[N:].max():5.2f} m,  roughness {roughness(slip_true[N:]):.3f}")
print(f"recovered:    peak {raw.dip_slip.max():5.2f} m,  roughness {roughness(raw.dip_slip):.3f}")
print()
G_dip = G_full[:, N:]
print(f"condition number of G^T G : {np.linalg.cond(G_dip.T @ G_dip):.1e}")

The condition number is enormous, so the inverse amplifies the 1 cm of station
noise into metres of spurious, oscillating slip. The fit to the *data* is fine
(`plot.fit` below) — the model is simply not trustworthy.

In [ ]:
geodef.plot.fit(gnss.obs, raw.predicted,
                title="Observed vs. predicted (fits the noise well)")

## Checkpoint questions

**Why avoid explicitly forming $(G^TWG)^{-1}$?**

<details><summary>Answer</summary>



Factorizations or least-squares solvers are more stable and efficient.



</details>

**What does a tiny singular value mean physically?**

<details><summary>Answer</summary>



One combination of slip changes the data only weakly and is easily dominated by noise.



</details>

**Can reduced chi-squared near one prove the slip is right?**

<details><summary>Answer</summary>



No. It checks residual scale under the stated covariance and model family.



</details>


## Common mistakes

- Using formal uncertainties that omit model error makes weighting and chi-squared misleading.
- Reading an oscillatory, excellent-fit solution as geological structure is overfitting.
- Equating underdetermined with unsolvable ignores priors and constraints—but those assumptions must be stated.


## Recap

- WLS follows from a Gaussian covariance model and whitening.
- Small singular values amplify noise into unstable slip modes.
- Fit diagnostics assess data space, not uniqueness in model space.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/03_unregularized_inversion_solutions.ipynb`.

1. Double the noise and track peak slip and reduced chi-squared.
2. Remove stations until the system is underdetermined.
3. Whiten $G$ and $d$ by Cholesky factor and compare with `solve`.
4. Challenge: plot the smallest right singular vector and compare it with the largest recovered oscillation.


## Further reading

- Gauss (1823), least squares and optimality.
- Golub & Van Loan (2013), numerical linear algebra and SVD.
- Menke (2018), geophysical inverse theory.
